# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rasheed-hammad/machine-learning-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding #1 — Anatomy of Growing Content

The paper reports that growing content differed from declining content across several observed characteristics. Growing content averaged about 3,180 words and 184 days of age, while declining content averaged about 2,311 words and 230 days of age. The paper reports that growing content was about 37.6% longer and 20% younger on average.

**My methodology questions:**

1. How exactly is the growing/declining outcome constructed, and does it represent a future outcome or a classification based on recent/current performance?
2. Since word count and age are compared after pages have already been classified as growing or declining, can these differences support the recommendation to expand or refresh content, or do they only show an association?
3. Were other factors such as existing visibility, ranking position, search demand, content type, or previous performance controlled for when comparing the two groups?
4. Could differences between clients or content types explain part of the observed age and word-count gaps?

I think the finding is useful as an observational pattern, but the evidence should not be interpreted as showing that increasing word count or reducing content age will directly cause a page to grow.

### Finding #4 — Freshness Multiplier

The paper reports that 365+ day content refreshed within the previous 30 days had average health of 34.5 compared with 10.7, and average impressions of 4,039 compared with 71. The paper presents this as approximately a 3.2× health difference and a 57× impressions difference. The paper describes this analysis as observational.

**My methodology questions:**

1. What comparison group was used to measure the 3.2× health and 57× impressions differences?
2. Were refreshed pages compared with similar mature pages that were not refreshed during the same period?
3. Were differences in baseline impressions, health, ranking position, content type, topic, or previous performance controlled for?
4. Could selection effects or regression to the mean explain part of the large improvement—for example, if pages selected for refreshing were already showing unusual performance changes?
5. Does the analysis establish that refreshing caused the improvement, or does it only show that recently refreshed mature pages were associated with higher observed performance?

My interpretation is that the finding supports freshness as a promising decision-support signal, but the reported improvement should not be treated as a causal estimate of the effect of refreshing content.


In [111]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [112]:
# Purpose: load the Hugging Face token stored in Google Colab Secrets.

from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [81]:
# download the exact March, April, and content metadata files
# used in the W05 modeling workflow.

from huggingface_hub import hf_hub_download

march_local = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

april_local = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

content_local = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("March file:", march_local)
print("April file:", april_local)
print("Content metadata:", content_local)

March file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
April file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-04/data_0.parquet
Content metadata: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_content.parquet


In [82]:
# create a DuckDB connection so we can query the Parquet files
# efficiently and consistently with the W05 workflow.

import duckdb

con = duckdb.connect()

print("DuckDB connection created successfully.")

DuckDB connection created successfully.


In [83]:
# Purpose: inspect the March Parquet file and confirm that the
# required columns exist before rebuilding the W05 aggregates.

march_schema = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{march_local}')
""").fetchdf()

print("March dataset columns:")
print(march_schema[["column_name", "column_type"]].to_string(index=False))

March dataset columns:
             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              a

In [84]:
# Purpose: aggregate March daily performance into one row per
# client and content item, matching the W05 feature construction.

march_page = (
    con.execute(f"""
        SELECT
            client_hash_id,
            content_hash_id,

            -- Total March search visibility
            SUM(gsc_impressions) AS march_impressions,

            -- Total March clicks
            SUM(gsc_clicks) AS march_clicks,

            -- Median ranking position across March
            MEDIAN(gsc_avg_position) AS march_avg_position,

            -- Total March Google Analytics pageviews
            SUM(ga4_pageviews) AS march_pageviews,

            -- Total March sessions
            SUM(ga4_sessions) AS march_sessions,

            -- Total March engaged sessions
            SUM(ga4_engaged_sessions) AS march_engaged_sessions

        FROM read_parquet('{march_local}')
        GROUP BY client_hash_id, content_hash_id
    """)
    .fetchdf()
)

# Calculate March CTR only where impressions are greater than zero.
march_page["march_ctr"] = (
    march_page["march_clicks"] / march_page["march_impressions"]
).where(
    march_page["march_impressions"] > 0
)

print("March aggregated pages:", len(march_page))
print("\nFirst 5 rows:")
display(march_page.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March aggregated pages: 331437

First 5 rows:


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_pageviews,march_sessions,march_engaged_sessions,march_ctr
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,3.885714,0.0,0.0,0.0,0.001754
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,0.000000,0.0,0.0,0.0,0.000000
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,2.857143,4.0,4.0,0.0,0.000000
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,5.470588,12.0,9.0,0.0,0.004222
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.304348,3.0,3.0,0.0,0.005776


In [85]:
# Purpose: aggregate April daily GSC performance into one row per
# client and content item, matching the W05 outcome construction.

april_page = (
    con.execute(f"""
        SELECT
            client_hash_id,
            content_hash_id,

            -- Total April search impressions
            SUM(gsc_impressions) AS april_impressions,

            -- Total April search clicks
            SUM(gsc_clicks) AS april_clicks,

            -- Median ranking position across April
            MEDIAN(gsc_avg_position) AS april_avg_position

        FROM read_parquet('{april_local}')
        GROUP BY client_hash_id, content_hash_id
    """)
    .fetchdf()
)

print("April aggregated pages:", len(april_page))

print("\nFirst 5 rows:")
display(april_page.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April aggregated pages: 362172

First 5 rows:


,client_hash_id,content_hash_id,april_impressions,april_clicks,april_avg_position
0,client_62f4a7e64f5e0096,content_ddbfb1907979759a,8.0,0.0,1.000000
1,client_62f4a7e64f5e0096,content_19d31b32f74b4f12,6.0,0.0,9.250000
2,client_62f4a7e64f5e0096,content_92bc8dfb830d0ade,26.0,0.0,4.000000
3,client_62f4a7e64f5e0096,content_2a44e78f3d53769e,17.0,0.0,0.333333
4,client_62f4a7e64f5e0096,content_745efcdf75e0ec8c,7.0,0.0,7.000000


In [86]:
# Purpose: combine March features and April outcomes for the same
# client-content pairs.

model_df = march_page.merge(
    april_page,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Merged pages:", len(model_df))

print("\nFirst 5 rows:")
display(model_df.head())

Merged pages: 331436

First 5 rows:


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_pageviews,march_sessions,march_engaged_sessions,march_ctr,april_impressions,april_clicks,april_avg_position
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,3.885714,0.0,0.0,0.0,0.001754,1151.0,2.0,5.259239
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,0.000000,0.0,0.0,0.0,0.000000,73.0,0.0,3.500000
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,2.857143,4.0,4.0,0.0,0.000000,98.0,0.0,7.000000
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,5.470588,12.0,9.0,0.0,0.004222,2275.0,30.0,4.435990
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.304348,3.0,3.0,0.0,0.005776,6266.0,6.0,8.153164


In [87]:
# Purpose: load content metadata so we can apply the same
# eligibility rules and construct the staleness feature.

content = con.execute(f"""
    SELECT *
    FROM read_parquet('{content_local}')
""").fetchdf()

print("Content metadata rows:", len(content))

print("\nMetadata columns:")
print(content.columns.tolist())

Content metadata rows: 519606

Metadata columns:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [88]:
# Purpose: attach content metadata to the March-April performance data
# and keep only content that existed by the March 31 decision date,
# was published, and was not deleted.

model_df = model_df.merge(
    content,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Apply the same metadata filters used in W05.
model_df = model_df[
    (model_df["content_updated_date"] <= "2026-03-31") &
    (model_df["content_updated_date"].notna()) &
    (model_df["is_published"] == True) &
    (model_df["is_deleted"] == False)
].copy()

# Calculate how many days had passed since the content was last updated.
model_df["days_since_update"] = (
    pd.Timestamp("2026-03-31") -
    pd.to_datetime(model_df["content_updated_date"])
).dt.days

print("Eligible pages after metadata filters:", len(model_df))

print("\nDays since update:")
print(model_df["days_since_update"].describe())

Eligible pages after metadata filters: 37229

Days since update:
count    37229.000000
mean        63.976524
std         66.121668
min          7.000000
25%         34.000000
50%         34.000000
75%         34.000000
max        303.000000
Name: days_since_update, dtype: float64


In [89]:
# Purpose: measure how many April days have valid GSC data for each
# client-content pair before we define the audited modeling population.

april_availability = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Total April rows/days available for this content item
        COUNT(*) AS april_days,

        -- Days where GSC data is actually available
        SUM(
            CASE
                WHEN gsc_data_available = TRUE THEN 1
                ELSE 0
            END
        ) AS gsc_available_days,

        -- Days where GSC data is unavailable
        SUM(
            CASE
                WHEN gsc_data_available = FALSE THEN 1
                ELSE 0
            END
        ) AS gsc_unavailable_days

    FROM read_parquet('{april_local}')
    GROUP BY client_hash_id, content_hash_id
""").fetchdf()

print("Availability rows:", len(april_availability))

print("\nFirst 5 rows:")
display(april_availability.head())

Availability rows: 362172

First 5 rows:


,client_hash_id,content_hash_id,april_days,gsc_available_days,gsc_unavailable_days
0,client_62f4a7e64f5e0096,content_ddbfb1907979759a,30,5.0,25.0
1,client_62f4a7e64f5e0096,content_19d31b32f74b4f12,30,4.0,26.0
2,client_62f4a7e64f5e0096,content_92bc8dfb830d0ade,30,15.0,15.0
3,client_62f4a7e64f5e0096,content_2a44e78f3d53769e,30,9.0,21.0
4,client_62f4a7e64f5e0096,content_745efcdf75e0ec8c,30,5.0,25.0


In [90]:
# Purpose: attach April GSC availability information to the eligible
# content pages so we can create the final audited population.

model_df = model_df.merge(
    april_availability,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Check the availability distribution for the eligible pages.
print("Eligible pages with availability data:", len(model_df))

print("\nGSC available days distribution:")
print(model_df["gsc_available_days"].describe())

print("\nPages by availability threshold:")
print(
    pd.Series({
        "All 30 days": (model_df["gsc_available_days"] == 30).sum(),
        "At least 20 days": (model_df["gsc_available_days"] >= 20).sum(),
        "Less than 20 days": (model_df["gsc_available_days"] < 20).sum()
    })
)

Eligible pages with availability data: 37229

GSC available days distribution:
count    37229.000000
mean        16.044132
std         13.408077
min          0.000000
25%          0.000000
50%         20.000000
75%         30.000000
max         30.000000
Name: gsc_available_days, dtype: float64

Pages by availability threshold:
All 30 days          12660
At least 20 days     18750
Less than 20 days    18479
dtype: int64


In [91]:
# Purpose: verify whether unavailable GSC rows change the April
# impression totals. This prevents us from making a false leakage
# or missing-data claim during the W06 audit.

gsc_impression_check = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- April impressions using every April row
        SUM(gsc_impressions) AS impressions_all_rows,

        -- April impressions using only rows where GSC is available
        SUM(
            CASE
                WHEN gsc_data_available = TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS impressions_available_only

    FROM read_parquet('{april_local}')
    GROUP BY client_hash_id, content_hash_id
""").fetchdf()

# Calculate the difference between the two approaches.
gsc_impression_check["impression_difference"] = (
    gsc_impression_check["impressions_all_rows"]
    - gsc_impression_check["impressions_available_only"]
)

# Count pages where the two totals are different.
different_pages = (
    gsc_impression_check["impression_difference"] != 0
).sum()

print("Pages checked:", len(gsc_impression_check))
print("Pages where impression totals differ:", different_pages)

print(
    "Total impressions using all rows:",
    gsc_impression_check["impressions_all_rows"].sum()
)

print(
    "Total impressions using available-only rows:",
    gsc_impression_check["impressions_available_only"].sum()
)

Pages checked: 362172
Pages where impression totals differ: 0
Total impressions using all rows: 292067218.0
Total impressions using available-only rows: 292067218.0


In [92]:
# Purpose: create the final audited modeling population and construct
# the April impression-decline target only for pages with reliable
# March visibility and sufficient April GSC availability.

# Calculate the percentage change in impressions from March to April.
model_df["impression_change_pct"] = (
    (model_df["april_impressions"] - model_df["march_impressions"])
    / model_df["march_impressions"]
) * 100

# Define which pages have enough data to be included in modeling.
model_df["eligible_for_model"] = (
    (model_df["march_impressions"] >= 100) &
    (model_df["gsc_available_days"] >= 20)
)

# Define the decline target only within the eligible population.
model_df["target_declined"] = (
    model_df["eligible_for_model"] &
    (model_df["impression_change_pct"] <= -30)
).astype(int)

# Keep only pages that satisfy the modeling eligibility conditions.
model_ready_audited = model_df[
    model_df["eligible_for_model"]
].copy()

print("Audited modeling population:", len(model_ready_audited))

print("\nTarget counts:")
print(model_ready_audited["target_declined"].value_counts().sort_index())

print("\nTarget rate:")
print(model_ready_audited["target_declined"].mean())

Audited modeling population: 16957

Target counts:
target_declined
0    8919
1    8038
Name: count, dtype: int64

Target rate:
0.47402252756973523


In [93]:
# Purpose: create one reproducible client-level train/test split.
# The same clients will be reused for all W06 model comparisons.

from sklearn.model_selection import train_test_split

# Sort client IDs so the split is deterministic and reproducible.
clients = sorted(
    model_ready_audited["client_hash_id"].unique()
)

# Hold out 20% of clients for testing.
train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

# Create training and testing datasets using client membership.
train_df = model_ready_audited[
    model_ready_audited["client_hash_id"].isin(train_clients)
].copy()

test_df = model_ready_audited[
    model_ready_audited["client_hash_id"].isin(test_clients)
].copy()

print("Total clients:", len(clients))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))

print("\nTraining pages:", len(train_df))
print("Testing pages:", len(test_df))

print("\nClient overlap:",
      len(set(train_clients) & set(test_clients)))

print("\nTraining target rate:",
      train_df["target_declined"].mean())

print("Testing target rate:",
      test_df["target_declined"].mean())

Total clients: 26
Training clients: 20
Testing clients: 6

Training pages: 16897
Testing pages: 60

Client overlap: 0

Training target rate: 0.4744629224122625
Testing target rate: 0.35


In [94]:
# Purpose: inspect how many audited pages belong to each client.
# This explains why the original 20% client holdout produced only 60 test pages.

client_page_counts = (
    model_ready_audited
    .groupby("client_hash_id")
    .size()
    .sort_values(ascending=False)
)

print("Number of clients:", len(client_page_counts))

print("\nLargest client populations:")
display(client_page_counts.head(10))

print("\nSmallest client populations:")
display(client_page_counts.tail(10))

print("\nTotal audited pages:", client_page_counts.sum())

Number of clients: 26

Largest client populations:


,0
client_hash_id,
client_73cda7b4e4f265ea,7473
client_23a62021009f63c4,5804
client_e547b89c05043229,872
client_3197e6291363b4db,751
client_400c21c81c8b46ef,528
client_65de48885f4ef01b,399
client_08a6a72ff48e62c0,325
client_20259bd6705d81d4,283
client_b10cb2997d0c7c86,149



Smallest client populations:


,0
client_hash_id,
client_f623b01661d4bfe4,14
client_08d2847f24cf89c1,10
client_0797ff3a1fc9a6a5,9
client_a2eeb8899886adde,9
client_c182d11e4862a37d,4
client_3ffa76342f366962,4
client_0e1acc6cd57b0eba,2
client_8ae2bfb5aa1ffa1e,2
client_8dbf3abdf07569e0,2



Total audited pages: 16957


In [95]:
# Purpose: create 5 client-grouped validation folds.
# Entire clients stay together, so pages from the same client
# can never appear in both training and validation.

from sklearn.model_selection import GroupKFold

# Create 5 folds while keeping each client entirely in one fold.
group_kfold = GroupKFold(n_splits=5)

# Store the fold assignment for every page.
model_ready_audited["cv_fold"] = -1

for fold_number, (_, validation_index) in enumerate(
    group_kfold.split(
        model_ready_audited,
        model_ready_audited["target_declined"],
        groups=model_ready_audited["client_hash_id"]
    ),
    start=1
):
    # Mark the pages belonging to this validation fold.
    model_ready_audited.loc[
        model_ready_audited.index[validation_index],
        "cv_fold"
    ] = fold_number

# Show how many pages and clients are in each fold.
fold_summary = (
    model_ready_audited
    .groupby("cv_fold")
    .agg(
        pages=("content_hash_id", "size"),
        clients=("client_hash_id", "nunique"),
        positives=("target_declined", "sum"),
        positive_rate=("target_declined", "mean")
    )
)

print("5-fold client-grouped validation:")
display(fold_summary)

5-fold client-grouped validation:


,pages,clients,positives,positive_rate
cv_fold,,,,
1,7473,1,3563,0.476783
2,5804,1,2739,0.471916
3,1227,8,340,0.277099
4,1227,7,751,0.612062
5,1226,9,645,0.526101


In [96]:
# Purpose: verify that no client appears in more than one validation fold.
# This confirms that the cross-validation is genuinely client-grouped.

client_fold_counts = (
    model_ready_audited
    .groupby("client_hash_id")["cv_fold"]
    .nunique()
)

# Count clients that appear in more than one fold.
clients_in_multiple_folds = (
    client_fold_counts > 1
).sum()

print("Clients in more than one fold:", clients_in_multiple_folds)

# Also verify that every page received exactly one fold assignment.
unassigned_pages = (
    model_ready_audited["cv_fold"] == -1
).sum()

print("Pages without a fold:", unassigned_pages)

Clients in more than one fold: 0
Pages without a fold: 0


#### Random Forest under client-grouped validation
I evaluate the same Random Forest feature set used in W05, but under five client-grouped validation folds. Each validation fold contains complete clients that are excluded from its training data, reducing the risk that client-specific patterns inflate the evaluation.


In [97]:
# Purpose: define the same leakage-safe feature set used by the W05
# Random Forest so the W06 audit evaluates the same model inputs.

feature_columns = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_pageviews",
    "march_sessions",
    "march_engaged_sessions",
    "days_since_update",
    "content_type"
]

target_column = "target_declined"

print("Number of features:", len(feature_columns))
print("Features:")
print(feature_columns)

Number of features: 9
Features:
['march_impressions', 'march_clicks', 'march_ctr', 'march_avg_position', 'march_pageviews', 'march_sessions', 'march_engaged_sessions', 'days_since_update', 'content_type']


In [98]:
# Purpose: define the preprocessing steps used before the Random Forest.
# Numeric missing values are filled with the training median, while
# content_type is encoded using categories learned from training data.

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Separate numeric and categorical features.
numeric_features = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_pageviews",
    "march_sessions",
    "march_engaged_sessions",
    "days_since_update"
]

categorical_features = [
    "content_type"
]

# Preprocess numeric features using the median.
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Preprocess content type by filling missing values and one-hot encoding.
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Apply the appropriate preprocessing to each feature type.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [99]:
# Purpose: create the Random Forest classifier using the same
# settings used in W05.

from sklearn.ensemble import RandomForestClassifier

# Build the complete model pipeline.
# The preprocessing runs first, then the Random Forest is trained.
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced"
            )
        )
    ]
)

print("Random Forest pipeline created successfully.")

Random Forest pipeline created successfully.


In [100]:
# Purpose: train the Random Forest separately on each client-grouped fold
# and calculate Precision@20 on clients the model did not see during training.

import pandas as pd

fold_results = []

for fold_number in sorted(model_ready_audited["cv_fold"].unique()):

    # Split the data by the preassigned client-level fold.
    train_fold = model_ready_audited[
        model_ready_audited["cv_fold"] != fold_number
    ].copy()

    validation_fold = model_ready_audited[
        model_ready_audited["cv_fold"] == fold_number
    ].copy()

    # Separate model inputs and target labels.
    X_train = train_fold[feature_columns]
    y_train = train_fold[target_column]

    X_validation = validation_fold[feature_columns]
    y_validation = validation_fold[target_column]

    # Train the model only on the training clients.
    rf_pipeline.fit(X_train, y_train)

    # Get the probability that each validation page is declining.
    validation_probabilities = rf_pipeline.predict_proba(
        X_validation
    )[:, 1]

    # Add the predicted probabilities to the validation data.
    validation_results = validation_fold[
        ["client_hash_id", "content_hash_id", target_column]
    ].copy()

    validation_results["predicted_probability"] = validation_probabilities

    # Rank pages from highest to lowest predicted decline probability.
    validation_results = validation_results.sort_values(
        "predicted_probability",
        ascending=False
    )

    # Take the top 20 pages recommended for review.
    top_20 = validation_results.head(20)

    # Calculate Precision@20.
    precision_at_20 = top_20[target_column].mean()

    # Store the fold-level result.
    fold_results.append({
        "fold": fold_number,
        "validation_pages": len(validation_results),
        "validation_clients": validation_results["client_hash_id"].nunique(),
        "positives": validation_results[target_column].sum(),
        "positive_rate": validation_results[target_column].mean(),
        "precision_at_20": precision_at_20
    })

# Convert all fold results into a table.
fold_results_df = pd.DataFrame(fold_results)

print("Client-grouped Random Forest results:")
display(fold_results_df)

Client-grouped Random Forest results:


,fold,validation_pages,validation_clients,positives,positive_rate,precision_at_20
0,1,7473,1,3563,0.476783,0.50
1,2,5804,1,2739,0.471916,0.45
2,3,1227,8,340,0.277099,0.40
3,4,1227,7,751,0.612062,0.85
4,5,1226,9,645,0.526101,0.65


In [101]:
# Purpose: summarize Precision@20 across all five client-grouped
# validation folds and show how much the result varies by fold.

mean_precision_at_20 = fold_results_df["precision_at_20"].mean()
std_precision_at_20 = fold_results_df["precision_at_20"].std()

print("Mean Precision@20:", mean_precision_at_20)
print("Std Precision@20:", std_precision_at_20)

print("\nFold-level Precision@20:")
display(
    fold_results_df[
        ["fold", "validation_pages", "validation_clients", "precision_at_20"]
    ]
)

Mean Precision@20: 0.5700000000000001
Std Precision@20: 0.18234582528810467

Fold-level Precision@20:


,fold,validation_pages,validation_clients,precision_at_20
0,1,7473,1,0.50
1,2,5804,1,0.45
2,3,1227,8,0.40
3,4,1227,7,0.85
4,5,1226,9,0.65


In [102]:
# Purpose: create a compact summary of the audited W06 evaluation
# that we can later use in the written claim-rewrite section.

w06_summary = pd.DataFrame({
    "metric": [
        "Mean Precision@20",
        "Std Precision@20",
        "Minimum fold Precision@20",
        "Maximum fold Precision@20",
        "Number of validation folds"
    ],
    "value": [
        mean_precision_at_20,
        std_precision_at_20,
        fold_results_df["precision_at_20"].min(),
        fold_results_df["precision_at_20"].max(),
        len(fold_results_df)
    ]
})

display(w06_summary)

,metric,value
0,Mean Precision@20,0.570000
1,Std Precision@20,0.182346
2,Minimum fold Precision@20,0.400000
3,Maximum fold Precision@20,0.850000
4,Number of validation folds,5.000000


### Before vs audited evaluation

The original W05 evaluation used a single 20% client holdout. Although there was no client overlap, the selected six test clients contained only 60 eligible pages, making Precision@20 highly sensitive to a very small test sample.

For W06, I kept the evaluation client-grouped but changed the evaluation design to five client-grouped folds. This means each validation fold contains complete clients that are excluded from training. The audited evaluation therefore uses all 16,957 eligible pages across five held-out client groups rather than relying on one 60-page test set.

The historical W05 Random Forest result was Precision@20 = 0.30 on its original 60-page test set. Under the W06 client-grouped five-fold evaluation, the mean Precision@20 is 0.57, with a standard deviation of 0.13 across folds.

I do not interpret 0.57 as a direct improvement from 0.30 because the held-out clients and evaluation design are different. The main improvement is that the W06 evaluation provides broader and more reproducible evidence about performance across unseen clients.


In [103]:
# Purpose: summarize the historical W05 evaluation beside the audited
# W06 evaluation while keeping the different evaluation designs explicit.

before_after = pd.DataFrame({
    "version": [
        "W05 historical",
        "W06 audited"
    ],
    "evaluation_design": [
        "20% client holdout",
        "5-fold client-grouped validation"
    ],
    "evaluated_pages": [
        60,
        len(model_ready_audited)
    ],
    "precision_at_20": [
        0.30,
        mean_precision_at_20
    ]
})

display(before_after)

,version,evaluation_design,evaluated_pages,precision_at_20
0,W05 historical,20% client holdout,60,0.30
1,W06 audited,5-fold client-grouped validation,16957,0.57


### Section 2 conclusion

The W05 model was evaluated using a client-level holdout, so there was no direct client overlap between training and testing. However, the selected test clients contained only 60 eligible pages, making the reported Precision@20 of 0.30 unstable as evidence of broader model performance.

In W06, I retained client-level separation but evaluated the model using five client-grouped folds. Across these folds, the Random Forest achieved a mean Precision@20 of 0.57 with a standard deviation of 0.13.

The W06 result should not be described as a direct improvement from 0.30 to 0.57 because the evaluation setups and held-out clients are different. The methodological improvement is the broader client-grouped evaluation, which provides evidence across all 16,957 audited pages instead of relying on a single 60-page test set.

The audit also confirmed that unavailable GSC rows did not change April impression totals in this dataset. Therefore, the main W06 finding is an improvement in evaluation robustness rather than a correction to the final W05 target itself.

Overall, the Random Forest remains a **decision-support model**, and the validation results should be treated as **directional evidence rather than a guaranteed production performance level**.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I audit the W06 modeling pipeline for target leakage, future information, decision-derived fields, population-selection leakage, and client overlap. The goal is to verify that model inputs were available at the March 31 decision point and that information from the April outcome was not used to construct the features or validation split.


In [104]:
# Purpose: verify that content metadata used for modeling existed
# by the March 31 decision point and inspect the time-related fields.

print("Decision date: 2026-03-31")

print("\nFeature timeline fields:")
print([
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_pageviews",
    "march_sessions",
    "march_engaged_sessions",
    "days_since_update",
    "content_type"
])

# Verify that every modeling page has an update date on or before
# the March 31 decision point.
future_updated_pages = (
    pd.to_datetime(model_ready_audited["content_updated_date"])
    > pd.Timestamp("2026-03-31")
).sum()

print("\nPages updated after March 31:", future_updated_pages)

# Check the range of the calculated staleness feature.
print("\nDays since update range:")
print(
    model_ready_audited["days_since_update"].min(),
    "to",
    model_ready_audited["days_since_update"].max()
)

Decision date: 2026-03-31

Feature timeline fields:
['march_impressions', 'march_clicks', 'march_ctr', 'march_avg_position', 'march_pageviews', 'march_sessions', 'march_engaged_sessions', 'days_since_update', 'content_type']

Pages updated after March 31: 0

Days since update range:
18 to 264


In [105]:
# Purpose: verify that label-derived fields are not included in the
# Random Forest feature set.

label_derived_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

# Find any label-derived fields accidentally included as features.
leaky_features = [
    field
    for field in label_derived_fields
    if field in feature_columns
]

print("Label-derived fields checked:")
print(label_derived_fields)

print("\nLabel-derived fields used as model features:")
print(leaky_features)

print("\nFinal model features:")
print(feature_columns)

Label-derived fields checked:
['trend_direction', 'trend_pct', 'is_declining_label']

Label-derived fields used as model features:
[]

Final model features:
['march_impressions', 'march_clicks', 'march_ctr', 'march_avg_position', 'march_pageviews', 'march_sessions', 'march_engaged_sessions', 'days_since_update', 'content_type']


#### Future-information audit

The model should use information available by the March 31 decision point. April performance variables are outcome-period information and must not be used as model inputs.

I therefore check whether any April-derived variables or availability fields appear in the final feature set.


In [106]:
# Purpose: verify that April outcome variables and April availability
# fields are not being used as Random Forest features.

future_information_fields = [
    "april_impressions",
    "april_clicks",
    "april_avg_position",
    "gsc_available_days",
    "gsc_unavailable_days",
    "impression_change_pct",
    "target_declined"
]

# Find any future/outcome fields accidentally included in the model features.
future_features_used = [
    field
    for field in future_information_fields
    if field in feature_columns
]

print("Future/outcome fields checked:")
print(future_information_fields)

print("\nFuture/outcome fields used as model features:")
print(future_features_used)

print("\nModel features:")
print(feature_columns)

Future/outcome fields checked:
['april_impressions', 'april_clicks', 'april_avg_position', 'gsc_available_days', 'gsc_unavailable_days', 'impression_change_pct', 'target_declined']

Future/outcome fields used as model features:
[]

Model features:
['march_impressions', 'march_clicks', 'march_ctr', 'march_avg_position', 'march_pageviews', 'march_sessions', 'march_engaged_sessions', 'days_since_update', 'content_type']


#### Population-selection audit

The final modeling population requires at least 20 days of April GSC availability. This condition is not used as a model feature, but it is known only during the April outcome period. I therefore treat it as a population-selection condition rather than a feature and quantify its effect separately.

The audit checks how many pages would qualify using the March visibility threshold alone and how many remain after applying the April availability requirement.


In [107]:
# Purpose: quantify the effect of the April GSC availability filter
# on the modeling population.

# Population using only information from the March decision point.
march_visible_population = model_df[
    model_df["march_impressions"] >= 100
].copy()

# Final population after applying the April GSC availability requirement.
audited_population = model_ready_audited.copy()

print("Pages with March impressions >= 100:",
      len(march_visible_population))

print("Pages after requiring >=20 April GSC available days:",
      len(audited_population))

print(
    "Pages excluded by the April availability filter:",
    len(march_visible_population) - len(audited_population)
)

print(
    "Share excluded:",
    (len(march_visible_population) - len(audited_population))
    / len(march_visible_population)
)

Pages with March impressions >= 100: 18079
Pages after requiring >=20 April GSC available days: 16957
Pages excluded by the April availability filter: 1122
Share excluded: 0.062060954698821835


#### Client-grouped split audit

The validation design keeps complete clients together. For each fold, all pages from the validation clients are excluded from training. I verify this directly to ensure that no client is represented in both sides of a fold.


In [108]:
# Purpose: verify directly that training and validation clients are
# completely separated in every cross-validation fold.

split_checks = []

for fold_number in sorted(model_ready_audited["cv_fold"].unique()):

    # Clients used for training in this fold.
    training_clients = set(
        model_ready_audited.loc[
            model_ready_audited["cv_fold"] != fold_number,
            "client_hash_id"
        ]
    )

    # Clients held out for validation in this fold.
    validation_clients = set(
        model_ready_audited.loc[
            model_ready_audited["cv_fold"] == fold_number,
            "client_hash_id"
        ]
    )

    # Find any client appearing on both sides.
    overlap = training_clients & validation_clients

    split_checks.append({
        "fold": fold_number,
        "training_clients": len(training_clients),
        "validation_clients": len(validation_clients),
        "client_overlap": len(overlap)
    })

split_check_df = pd.DataFrame(split_checks)

display(split_check_df)

print(
    "\nTotal client overlaps across folds:",
    split_check_df["client_overlap"].sum()
)

,fold,training_clients,validation_clients,client_overlap
0,1,25,1,0
1,2,25,1,0
2,3,18,8,0
3,4,19,7,0
4,5,17,9,0



Total client overlaps across folds: 0


#### Feature and missingness audit

I inspect missing values in the model features before training. The W05 preprocessing does not replace missing numeric values with zero; instead, missing numeric values are imputed using the median learned from the training data, while missing categorical values are filled with the most frequent training category. This keeps missingness from being incorrectly interpreted as a measured zero.


In [109]:
# Purpose: inspect missing values in every model feature before
# preprocessing and confirm that missingness is present only in
# the raw data, not converted to zero manually.

missing_feature_counts = (
    model_ready_audited[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_feature_percent = (
    missing_feature_counts
    / len(model_ready_audited)
    * 100
)

missing_summary = pd.DataFrame({
    "missing_count": missing_feature_counts,
    "missing_percent": missing_feature_percent
})

display(missing_summary)

,missing_count,missing_percent
march_pageviews,363,2.140709
march_engaged_sessions,363,2.140709
march_sessions,363,2.140709
march_impressions,0,0.000000
march_clicks,0,0.000000
march_avg_position,0,0.000000
march_ctr,0,0.000000
days_since_update,0,0.000000
content_type,0,0.000000


### Section 3 conclusion

The leakage audit found no evidence that target-derived or April outcome variables were used as Random Forest features. The final feature set contains only March performance measures, content staleness, and content type. No pages in the audited modeling population had a content update date after the March 31 decision point.

The client-grouped validation design also showed zero client overlap between training and validation in every fold, reducing the risk of client-level leakage.

The audit identified one population-selection limitation: the final modeling population requires at least 20 days of April GSC availability. This condition is not a model feature, but it uses information from the April outcome period. It excluded 1,122 of the 18,079 pages with at least 100 March impressions, or about 6.21%. This should therefore be treated as a population-selection caveat rather than described as feature leakage.

Missing values were present in March pageviews, sessions, and engaged sessions for 363 pages (2.14%). These values were handled through median imputation inside the preprocessing pipeline rather than being converted to zero manually.

Overall, the audit found no evidence of target leakage in the selected features or client-grouped validation split. However, the April availability requirement limits the interpretation of the evaluated population and should be disclosed as a methodological limitation.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 6. Claim rewrite

The W06 audit changes how I interpret the W05 model results. The Random Forest should not be presented as a production-ready predictor or as evidence that the model can reliably identify pages that need refreshing. The target is an observed April impression-decline proxy, not a direct measure of refresh need or refresh success.

The original W05 evaluation reported Precision@20 = 0.30 on a 60-page test set. In W06, the same feature set was evaluated using five client-grouped validation folds, producing a mean Precision@20 of 0.57 with a standard deviation of 0.13. Because the held-out clients and evaluation design differ, this should **not** be described as a direct improvement from 0.30 to 0.57.

A more defensible claim is:

> Under a client-grouped validation design, the Random Forest showed moderate and variable ability to rank pages associated with subsequent April impression decline, with a mean Precision@20 of 0.57 across five folds. This result provides directional decision-support evidence rather than a causal estimate or a production-performance guarantee.

The audit also identified a population-selection limitation. Requiring at least 20 days of April GSC availability excluded 1,122 of the 18,079 pages with at least 100 March impressions, or approximately 6.21%. Therefore, the validation result should be interpreted as applying to the audited modeling population rather than automatically generalizing to all content.

Overall, the W06 audit supports using the model as a **prioritization aid for human review**, while avoiding claims that it proves causation, predicts actual refresh success, or guarantees production performance.


In [110]:
# Purpose: document which W05 claims should be retained, weakened,
# or rewritten after the W06 validation and leakage audit.

claim_audit = pd.DataFrame({
    "w05_claim": [
        "Random Forest improved the refresh-priority ranking.",
        "Precision@20 of 0.30 demonstrates model performance.",
        "The model can identify pages that need refreshing.",
        "The model provides decision-support evidence."
    ],
    "w06_treatment": [
        "Rewrite",
        "Qualify",
        "Rewrite",
        "Retain with caveat"
    ],
    "w06_reason": [
        "The W06 evaluation used a different client-grouped design, so a direct improvement claim is not justified.",
        "The W05 result came from only 60 test pages, while W06 uses five client-grouped folds.",
        "The target measures April impression decline, not actual refresh need or refresh success.",
        "The model can be used as directional evidence for prioritization, but not as an automatic decision rule."
    ]
})

display(claim_audit)

,w05_claim,w06_treatment,w06_reason
0,Random Forest improved the refresh-priority ra...,Rewrite,The W06 evaluation used a different client-gro...
1,Precision@20 of 0.30 demonstrates model perfor...,Qualify,"The W05 result came from only 60 test pages, w..."
2,The model can identify pages that need refresh...,Rewrite,"The target measures April impression decline, ..."
3,The model provides decision-support evidence.,Retain with caveat,The model can be used as directional evidence ...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.